In [ ]:
from bak.datasets import StrawberryDataset
from matplotlib import pyplot as plt

data_ds = StrawberryDataset(r"strawberry_cls\images")
image, labels = data_ds[0]

print(image.max(), image.min())

plt.imshow(image.permute(1, 2, 0))
plt.axis('off')
plt.show()

# for label in labels:
#     class_id, x_min, y_min, x_max, y_max = label
#     x_min, y_min, x_max, y_max = list(map(lambda x: x * 640, [x_min, y_min, x_max, y_max]))
#     plt.gca().add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
#                                       edgecolor='red', facecolor='none', linewidth=2))
#     plt.text(x_min, y_min - 10, str(class_id), color='red', fontsize=12)
# plt.show()

In [ ]:
# def polygan2box(text, format='xyxy'):
#     text = text.replace("\n", "").split(" ")
#     class_id = int(text[0])
#     poly = list(map(float, text[1:]))
    
#     xs = poly[::2]
#     ys = poly[1::2]
#     x_min, x_max = min(xs), max(xs)
#     y_min, y_max = min(ys), max(ys)
#     if format == 'xyxy':
#         return [class_id, x_min, y_min, x_max, y_max]
#     elif format == 'xywh':
#         return [class_id, x_min, y_min, x_max - x_min, y_max - y_min]
#     elif format == 'cxcywh':
#         return [class_id, (x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min]

# import glob 
# import os

# labels = glob.glob("strawberry_cls/labels.init/*.txt")
# new_folder = "strawberry_cls/labels"

# os.makedirs(new_folder, exist_ok=True)

# for label in labels:
#     with open(label, "r") as f:
#         lines = f.readlines()
        
#     new_label_path = os.path.join(new_folder, os.path.basename(label))
#     with open(new_label_path, "w") as f:
#         for line in lines:
#             box = polygan2box(line, format='cxcywh')
#             f.write(" ".join(map(str, box)) + "\n")
            

In [ ]:
# import os
# import glob
# import random
# import shutil
# from os import path as osp

# ds_path = "strawberry_cls/"

# train_path = osp.join(ds_path, "train")
# val_path = osp.join(ds_path, "val")
# test_path = osp.join(ds_path, "test")

# for path in [train_path, val_path, test_path]:
#     os.makedirs(path, exist_ok=True)
#     os.makedirs(osp.join(path, "images"), exist_ok=True)
#     os.makedirs(osp.join(path, "labels"), exist_ok=True)

# images = glob.glob(osp.join(ds_path, "images/*.jpg"))
# random.shuffle(images)
# num_images = len(images)
# num_train = int(0.7 * num_images)
# num_val = int(0.15 * num_images)

# train_images = images[:num_train]
# val_images = images[num_train:num_train + num_val]
# test_images = images[num_train + num_val:]

# for img_path in train_images:
#     shutil.copy(img_path, osp.join(train_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(train_path, "labels"))
    
# for img_path in val_images:
#     shutil.copy(img_path, osp.join(val_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(val_path, "labels"))

# for img_path in test_images:
#     shutil.copy(img_path, osp.join(test_path, "images"))
#     shutil.copy(img_path.replace("images", "labels").replace(".jpg", ".txt"), osp.join(test_path, "labels"))


In [ ]:
import yaml
import torch

from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import ops

d = yaml.load(open("yolov12.yaml", "r"), Loader=yaml.FullLoader)
d["scale"] = "n"
model = DetectionModel(cfg=d, ch=3, verbose=False)

model_in = torch.randn(1, 3, 640, 640)

# train model only outputs predictions
model.train()
out = model(model_in)
print(type(out))
for item in out:
    print(type(item), item.shape)

# eval model outputs predictions and features
model.eval()
pred, feature = model(model_in)
print(type(pred), type(feature))
print(pred.shape)
for item in feature:
    print(type(item), item.shape)

# NMS
out = ops.non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
print(type(out), len(out))
for item in out:
    print(type(item), item.shape)

#### **测试代码**

In [ ]:
import glob
import yaml
import torch
import numpy as np

from tqdm import tqdm
from torch.utils.data import DataLoader

from bak.datasets import StrawberryDataset
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import ops
from types import SimpleNamespace
from metrics import mean_ap

device = torch.device("cpu")

default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", 'r'), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False).to(device)
model.args = SimpleNamespace(**{**default_cfg, **model_cfg})
# model.load_state_dict(torch.load("ckpts/yolov12.pt", map_location=device)["model"])
model.eval()

images = glob.glob("strawberry_cls/images/*.jpg")
np.random.shuffle(images)
test_images = images[int(0.8 * len(images)):]
print(f"Test images: {len(test_images)}")
data_ds = StrawberryDataset(test_images)
data_dl = DataLoader(data_ds, batch_size=1, shuffle=False, collate_fn=StrawberryDataset.collate_fn)


pred_list = []
target_list = []

for batch in tqdm(data_dl, desc="Testing", dynamic_ncols=True):
    model_in = batch["img"].to(device)
    with torch.no_grad():
        pred, _ = model(model_in)
    pred = ops.non_max_suppression(pred, conf_thres=0.25, iou_thres=0.45)
    
    batch_size = model_in.shape[0]
    img_h, img_w = model_in.shape[2], model_in.shape[3]

    for i in range(batch_size):
        pred_i = pred[i].detach().cpu().numpy() if len(pred) > i else np.zeros((0, 6), dtype=np.float32)
        pred_list.append(pred_i)

        idx = batch["batch_idx"] == i
        cls = batch["cls"][idx].detach().cpu().numpy().reshape(-1)
        bboxes = batch["bboxes"][idx].detach().cpu().numpy()

        if len(bboxes):
            bboxes_xyxy = ops.xywh2xyxy(torch.from_numpy(bboxes)).numpy()
            bboxes_xyxy[:, [0, 2]] *= img_w
            bboxes_xyxy[:, [1, 3]] *= img_h
            targets_i = np.column_stack([cls, bboxes_xyxy]).astype(np.float32)
        else:
            targets_i = np.zeros((0, 5), dtype=np.float32)

        target_list.append(targets_i)

    num_classes = 3
    metrics = mean_ap(pred_list, target_list, num_classes=num_classes)

print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1-score: {metrics['f1']:.4f}")
print(f"mAP@0.5: {metrics['map50']:.4f}")
print(f"mAP@0.5:0.95: {metrics['map50_95']:.4f}")

In [ ]:
import yaml
# import torch

from ultralytics.data.build import build_yolo_dataset, build_dataloader
from torch.utils.data import DataLoader

from types import SimpleNamespace
default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", 'r'), Loader=yaml.FullLoader)
data_cfg = yaml.load(open("strawberry_cls/fruit.yaml", "r"), Loader=yaml.FullLoader)
cfg = SimpleNamespace(**default_cfg)

batch = 32
train_ds = build_yolo_dataset(cfg, "strawberry_cls/train/images", batch=batch, data=data_cfg, mode="train")
val_ds = build_yolo_dataset(cfg, "strawberry_cls/val/images", batch=batch, data=data_cfg, mode="val")
test_ds = build_yolo_dataset(cfg, "strawberry_cls/test/images", batch=batch, data=data_cfg, mode="test")
